In [ ]:
import numpy as np
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
import random
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import umap
import pandas as pd
from numpy import inf
import time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

In [ ]:
import opfython.math.general as g
import opfython.stream.parser as p
import opfython.stream.splitter as s
from opfython.models import SupervisedOPF
from opfython.stream import loader
from opfython.utils import logging

Load Dataset

In [ ]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')
df_algarrobo = pd.read_csv(r'../data/data_temp/algarrobo.csv')
covering_array  = np.loadtxt(r'../data/coveringArray.csv', delimiter=",", dtype=int)

df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]
X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())

In [ ]:
def cafs(ca,dataset_x,dataset_y, max_iter ,model,print_logs=False):

  global_data_set = dataset_x
  global_max = 0
  max_iteartion = 0
  result_list_x = []
  result_list_y = []
  result_list_score = []
  num_rows = ca.shape[0]

  if len(dataset_x.columns) < ca.shape[1] :
    num_colums = len(dataset_x.columns)
  else:
    num_colums = ca.shape[1]

  initialTestTraining(result_list_score,result_list_x, result_list_y,dataset_x, dataset_y,model)
  while max_iteartion < max_iter:

     lst_headers = global_data_set.columns.values.copy()
     max_score = 0.0
     mx_data_set = None
     #random.shuffle(lst_headers)
    
     for i in range(0,num_rows):
        lst_headers_to_select = []
        for j in range(0,num_colums ):
          if ca[i][j] == 1 :
              lst_headers_to_select.append(lst_headers[j])
        #with the list of headers to select get sub dat set of col with pandas
        if len(lst_headers_to_select) == 0:
            continue
        df_temp = dataset_x[lst_headers_to_select]
        x_train_temp, x_test_temp, y_train_temp, y_test_temp = train_test_split(dataset_x[lst_headers_to_select].values, dataset_y.values.ravel(),test_size=0.20,random_state=42)
        #train a model
        learning_alg = train_model(model)
        learning_alg.fit(x_train_temp, y_train_temp)
        y_pred = learning_alg.predict(x_test_temp)
        #get accuracy
        score = f1_score(y_test_temp, y_pred,average='macro')
        # if accuracy is better than maxScore so far assing subDataSet  to local_sub_data_set
        if score >= max_score :
            max_score = score
            mx_data_set = df_temp.copy()

     global_data_set = mx_data_set.copy()
     global_max = max_score
     if print_logs:
         print(f"best f1 score= {global_max}, iteration:{max_iteartion}, numbers features selected ={len(global_data_set.columns)},best features selected={', '.join(global_data_set)}" )

     num_colums  = len(global_data_set.columns)
     mx_data_set = None
     max_score = 0
     max_iteartion = max_iteartion  +1
     result_list_x.append(max_iteartion)
     result_list_y.append(len(global_data_set.columns))
     result_list_score.append(global_max)
  return result_list_x,result_list_y,result_list_score

def initialTestTraining(score_list, iter_list, feature_list ,X,y,model):
    X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(X.values, y.values.ravel(), test_size=0.20, random_state=42)
    clf_new = train_model(model)
    clf_new.fit(X_train_temp, y_train_temp)
    y_pred = clf_new.predict(X_test_temp)
    score_list.append(f1_score(y_test_temp, y_pred,average='macro'))
    feature_list.append(X.shape[1])
    iter_list.append(0)

def train_model(model_name='ComplementNB'):  
    m = eval(model_name)
    return m

In [ ]:
def  plot_results_for_covering_array(scores,feature,num_of_iterarion, path_to_save_image):
        color = 'tab:blue'
        res_scores = np.array(scores)
        res_features = np.array(feature)
        res_iter = np.array(num_of_iterarion)

        plt.figure(figsize=(11, 10))
        
        fig, ax1 = plt.subplots()
        barwidth = 0.4
        color = 'tab:red'
        ax1.set_xlabel('Iterations')
        ax1.set_ylabel('Number of features', color=color)
        #ax1.set_title("ICAFS Feature selection on the Cacao dataset")
        ax1.spines['top'].set_visible(False)
        ax1.bar(res_iter-0.2, res_features, color=color, width=barwidth)
        ax1.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax1.set_ylim(1,max(res_features)+3)
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
        #for i in range(len(res_iter)):
        #    ax1.text(i+1-0.2,    res_features[i], res_features[i],rotation='vertical')
        for bar in ax1.patches:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height}', fontsize=10,
                    ha='center', va='bottom', rotation=90)
            
        ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
        color = 'tab:blue'
        ax2.set_ylabel('F1_score', color=color)
        ax2.bar(res_iter+0.2, res_scores, color=color, width=barwidth)
        ax2.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax2.set_ylim(min(res_scores)-0.001, max(res_scores)+0.001)

        fig.tight_layout()  # otherwise the right y-label is slightly clipped
        #for i in range(len(res_iter)):
        #    ax2.text(i+1, res_scores[i], f"{res_scores[i]:.3f}",rotation='vertical')

        for bar in ax2.patches:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height:.2f}', fontsize=10,
                    ha='center', va='bottom', rotation=90)

        #plt.title('ICAFS Feature selection for Cacao Dataset with OPF', y=-0.20)
        plt.gca().set_frame_on(False)
        plt.savefig(path_to_save_image)

CAFS KNN

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cacao,y_cacao,10,'KNeighborsClassifier(n_neighbors=2)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cacao_knn_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS CACAO 5 CV KNN

In [ ]:
clf = KNeighborsClassifier(n_neighbors=2)
X = X_cacao[['1225', '1322', '1559', '1936', '2296']]
res = cross_val_score(clf,X.values, y_cacao.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Cacao nibs mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )


In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_algarrobo,y_algarrobo,10,'KNeighborsClassifier(n_neighbors=2)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\algarrobo_knn_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS ALGARROBO KNN 5 CV

In [ ]:
clf = KNeighborsClassifier(n_neighbors=2)
X = X_algarrobo[['EXGR', 'MGRVI', 'RVI', 'DVI', 'EVI']]
res = cross_val_score(clf,X.values, y_algarrobo.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Algarrobo nibs mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

In [ ]:

umap_3d = umap.UMAP(n_components=3)

In [ ]:

from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


X_reduced_algarrobo =  algarrobo_x[['EXGR', 'MGRVI', 'RVI', 'DVI', 'EVI']]
X_reduced_standarized = StandardScaler().fit_transform(X_reduced_algarrobo)
algarrobo_umap_3d_reduced = umap_3d.fit_transform(X_reduced_standarized)

fig = plt.figure()
 
# syntax for 3-D projection
ax = plt.axes(projection ='3d')
 
# defining all 3 axis
z = algarrobo_umap_3d_reduced[:,0]
x = algarrobo_umap_3d_reduced[:,1]
y = algarrobo_umap_3d_reduced[:,2]

#plt.savefig(path_to_save_image)
# plotting
ax.scatter(x, y, z, c=[sns.color_palette()[x] for x in df_algarrobo.Labels.map({"N":0, "P":1})])
ax.set_title('3D Algarrobo dataset reduced with UMAP with CAFS KNN')
plt.savefig( r'.\\output_images\\umap_.cafs_knn.png')

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cis,y_cis,10,'KNeighborsClassifier(n_neighbors=2)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cis_knn_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS CIS KNN 5 CV

In [ ]:
clf = KNeighborsClassifier(n_neighbors=2)
X = X_cis[['X', 'Y']]
res = cross_val_score(clf,X.values, y_cis.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CIS nibs mean:{res.mean()} , 5-CV CIS STD:{res.std()}" )

CAFS OPF

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_algarrobo,y_algarrobo,10,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\algarrobo_opf_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS OPF CV Algarrobo

In [ ]:
#clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
#res = cross_val_score(clf,X.values, y_algarrobo.values.ravel(), cv=5,scoring = 'f1_macro')
#print(f"5-CV Algarronoao nibs mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5,shuffle=True, random_state =42)
X = X_algarrobo[['NGRDI', 'GBRI.1', 'MGRVI', 'DVI', 'REVI', 'REDVI']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_algarrobo.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_algarrobo.values[test_index]
   
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Algarrobo OPF mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )



In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cacao,y_cacao,10,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cacao_opf_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

5-CV CAFS CACAO OPF

In [ ]:
from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5, shuffle=True, random_state =42)
X = X_cacao[['1121', '1223', '1317', '1734', '1907']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_cacao.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_cacao.values[test_index]
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Cacao OPF mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )

Build Covering Array for CIS CAFS Problem

In [ ]:

from testflows.combinatorics import Covering

items = []
amter_to_test = {"X":[0,1],"Y":[0,1],"X10":[0,1],"Y10":[0,1],"X20":[0,1],"Y20":[0,1],"X30":[0,1],"Y30":[0,1],"X40":[0,1],"Y40":[0,1]}
generate_covering_array = Covering(paramter_to_test, strength=2)
for row in generate_covering_array.array:
   items_2 =[]
   for(key,value) in row.items():
        items_2.append(value)
   items.append(items_2)
ca_for_cis =  np.array(items)


In [ ]:

start_time = time.time()
iterarions,features,scores = cafs(ca_for_cis,X_cis,y_cis,3,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cis_opf_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

In [ ]:
from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5, shuffle=True, random_state =42)
X = X_cis[['Y', 'Y40']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_cis.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_cis.values[test_index]
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Cacao OPF mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )

CAFS SVC 

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cacao,y_cacao,10,'svm.SVC()',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cacao_svc_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS SVC CACAO 5 CV

In [ ]:
clf = svm.SVC()
X = X_cacao[['1223', '1583', '1957', '2313']]
res = cross_val_score(clf,X.values, y_cacao.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CACAO nibs mean:{res.mean()} , 5-CV CACAO STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_algarrobo,y_algarrobo,10,'svm.SVC()',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\algarrobo_svc_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS SVC 5 CV ALGARROBO

In [ ]:
clf = svm.SVC()
X = X_algarrobo[['Edge Red', 'GBRI.1', 'NDVI', 'EVI', 'REVI']]
res = cross_val_score(clf,X.values, y_algarrobo.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Algarrobo nibs mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cis,y_cis,10,'svm.SVC()',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cis_svc_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS SVC CIS 5CV

In [ ]:
clf = svm.SVC()
X = X_cis[['X', 'Y', 'X30', 'Y40']]
res = cross_val_score(clf,X.values, y_cis.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CIS nibs mean:{res.mean()} , 5-CV CIS STD:{res.std()}" )

CAFS MLP

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cacao,y_cacao,10,'MLPClassifier(solver="sgd", max_iter=7000, shuffle=False)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cacao_mlp_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS MLP CACO 5 CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=7000, shuffle=False)
X = X_cacao[['1119', '1189', '1196', '1236', '1265', '1275', '1284', '1295', '1341', '1425', '1532', '1591', '1660', '1741', '1785', '1871', '1922', '1940', '2030', '2125', '2165', '2439']]
res = cross_val_score(clf,X.values, y_cacao.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV cacao nibs mean:{res.mean()} , 5-CV cacao STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_algarrobo,y_algarrobo,10,'MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\algarrobo_mlp_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS MLP ALGARROBO 5 CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)
X = X_algarrobo[['G', 'NIR', 'EXG', 'MGRVI']]
res = cross_val_score(clf,X.values, y_algarrobo.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Algarrobo mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterarions,features,scores = cafs(covering_array,X_cis,y_cis,10,'MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)',True)
plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cis_mlp_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

CAFS CIS MLP 5CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)
X = X_cis[['X', 'Y', 'Y30', 'Y40']]
res = cross_val_score(clf,X.values, y_cis.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CIS mean:{res.mean()} , 5-CV CIS STD:{res.std()}" )